# Task 11 — Dự báo SSA trên dữ liệu khôi phục TimesNet (`SSA_TimesNet`)

Chạy từ trên xuống bằng CPU trên Google Colab hoặc máy cá nhân. Sử dụng tệp nhu cầu khôi phục `demand_timesnet.parquet` từ [Task 10](../Task10_TimesNet_Recovery/run_info.md) và tệp kiểm thử `eval.parquet` đã đối soát ở [Task 6](../Task06_Data_Preparation/results/20261113T082302850933Z/run_info.md). Notebook xác minh SHA-256 của cả hai tệp trước khi dự báo.

Công thức trọng số SSA, quy tắc lịch, phân nhóm `psd` theo doanh số gốc `sale_amount` của tập `train` và cách tính trung bình qua 7 ngày giữ nguyên theo [mã gốc `ssa_forecasting.py`, commit `8af694a`](https://github.com/Dingdong-Inc/frn-50k-baseline/blob/8af694a09a5a427d1d50648d348c022565a781f2/demand_forecasting/SSA/ssa_forecasting.py) và đồng bộ 100% với [Task 8 (`SSA_RawSales`)](../Task08_SSA_Raw_Sales/notebooks/08_ssa_raw_sales.ipynb).

In [1]:
from datetime import datetime, timezone
from hashlib import sha256
from pathlib import Path
import json
import os
import platform
import subprocess

import numpy as np
import pandas as pd
import pyarrow as pa

TIMESNET_DRIVE_ID = '1f9db-U9f9C9OFS-V2EcmQv4FIU3bguR3'
EXPECTED_SHA256 = {
    'demand_timesnet': 'c34f4eafe087d225865a624ef24d213af566496541458c3075f6e596d9f1d694',
    'eval': '61cfd3cb60d767d38ba9e640be64e982a15b2918ca0ba14fe0283c5a2eac6872',
}

try:
    from google.colab import drive
except ImportError:
    drive = None

if drive is not None:
    drive.mount('/content/drive')
    project_dir = Path('/content/drive/MyDrive/PTDLKD')
    timesnet_path = Path('/content/task11_data/demand_timesnet.parquet')
    timesnet_path.parent.mkdir(parents=True, exist_ok=True)
    if not timesnet_path.is_file():
        subprocess.run(['pip', 'install', '-q', 'gdown'], check=True)
        subprocess.run(['gdown', TIMESNET_DRIVE_ID, '-O', str(timesnet_path)], check=True)
    eval_path = project_dir / 'data/eval.parquet'
    output_dir = Path('/content/task11_outputs')
    task08_csv = None
else:
    cwd = Path.cwd().resolve()
    project_dir = next((p for p in (cwd, *cwd.parents)
                        if (p / 'Task11_SSA_TimesNet').is_dir()), cwd)
    default_demand = project_dir / 'demand_timesnet.parquet'
    if not default_demand.is_file() and (project_dir / 'tmp/demand_timesnet.parquet').is_file():
        default_demand = project_dir / 'tmp/demand_timesnet.parquet'
    timesnet_path = Path(os.environ.get('TASK11_DEMAND_PARQUET', default_demand))
    eval_path = Path(os.environ.get('TASK11_EVAL_PARQUET', project_dir / 'eval.parquet'))
    output_dir = Path(os.environ.get('TASK11_OUTPUT_DIR', project_dir / 'Task11_SSA_TimesNet/results'))
    task08_csv = project_dir / 'Task08_SSA_Raw_Sales/results/ssa_raw_sales_metrics.csv'

## 1. Xác nhận dữ liệu đầu vào (SHA-256 Task 10 & Task 6)

Kiểm tra dấu vân tay SHA-256 của `demand_timesnet.parquet` (khớp biên bản Task 10) và `eval.parquet` (khớp biên bản Task 6). Chỉ nạp các cột vô hướng cần thiết bằng PyArrow, tuyệt đối không ghi đè cột `sale_amount` gốc.

In [2]:
def sha256_file(path: Path) -> str:
    digest = sha256()
    with path.open('rb') as f:
        for block in iter(lambda: f.read(8 * 1024 * 1024), b''):
            digest.update(block)
    return digest.hexdigest()

for key, path, ref_label in (
    ('demand_timesnet', timesnet_path, 'Task 10'),
    ('eval', eval_path, 'Task 6'),
):
    if not path.is_file():
        raise FileNotFoundError(f'Không tìm thấy {path}.')
    obs = sha256_file(path)
    if obs != EXPECTED_SHA256[key]:
        raise ValueError(f'SHA-256 của {path.name} ({obs}) không khớp {ref_label} ({EXPECTED_SHA256[key]}).')

inputs = {
    'demand_timesnet': {
        'filename': timesnet_path.name,
        'sha256': EXPECTED_SHA256['demand_timesnet'],
        'bytes': timesnet_path.stat().st_size,
    },
    'eval': {
        'filename': eval_path.name,
        'sha256': EXPECTED_SHA256['eval'],
        'bytes': eval_path.stat().st_size,
    },
}

COLS_EVAL = ['store_id', 'product_id', 'dt', 'sale_amount', 'stock_hour6_22_cnt',
             'discount', 'holiday_flag', 'precpt']
COLS_TRAIN = COLS_EVAL + ['sale_amount_pred']

train_df = pd.read_parquet(timesnet_path, columns=COLS_TRAIN, engine='pyarrow')
eval_df = pd.read_parquet(eval_path, columns=COLS_EVAL, engine='pyarrow')
if (len(train_df), len(eval_df)) != (4_500_000, 350_000):
    raise ValueError('Số dòng train/eval không khớp quy chuẩn 4.500.000 / 350.000.')

inputs['demand_timesnet']['rows'] = len(train_df)
inputs['eval']['rows'] = len(eval_df)
print(f"demand_timesnet.parquet : SHA-256 khớp Task 10 ({inputs['demand_timesnet']['bytes']:,} bytes, {len(train_df):,} dòng).")
print(f"eval.parquet            : SHA-256 khớp Task 6 ({inputs['eval']['bytes']:,} bytes, {len(eval_df):,} dòng).")

demand_timesnet.parquet : SHA-256 khớp Task 10 (81,562,029 bytes, 4,500,000 dòng).
eval.parquet            : SHA-256 khớp Task 6 (5,451,872 bytes, 350,000 dòng).


## 2. Dự báo `SSA_TimesNet` và đánh giá trên 206.562 dòng `eval` hợp lệ

- Gán `eval_df['sale_amount_pred'] = np.nan` cho 7 ngày dự báo rồi ghép nối với 90 ngày lịch sử.
- Đưa `sale_amount_pred` vào vị trí thứ 3 của mảng đặc trưng liên tục `x_cont` để SSA dự báo trên chuỗi nhu cầu đã khôi phục.
- Tính `psd` từ `sale_amount` **gốc** của tập `train` (`dt <= '2024-06-25'`) để phân nhóm `Low Sale` (`psd < 1`) và `High Sale` (`psd >= 1`) trùng khớp 100% với Task 8.
- Đánh giá đối chiếu `sale_amount` thật của `eval` trên các dòng `stock_hour6_22_cnt == 0`.

In [3]:
eval_df['sale_amount_pred'] = np.nan
all_data = pd.concat([train_df, eval_df], axis=0, ignore_index=True)
del train_df, eval_df

all_data = all_data.sort_values(by=['store_id', 'product_id', 'dt']).reset_index(drop=True)

dt_series = pd.to_datetime(all_data['dt'])
all_data['time_idx'] = (dt_series - dt_series.min()).dt.days
all_data['dow'] = dt_series.dt.dayofweek
all_data.loc[all_data['dt'].isin(['2024-05-02', '2024-05-03', '2024-05-04']), 'holiday_flag'] = 0
all_data.loc[all_data['holiday_flag'] == 1, 'dow'] = 5
all_data.loc[(all_data['holiday_flag'] == 0) & (all_data['dow'].isin([5, 6])), 'dow'] = 0
all_data['is_precpt'] = (all_data['precpt'] > 3.5).astype(int)

x_cont = all_data[['time_idx', 'discount', 'sale_amount_pred']].values.reshape(-1, 97, 3)
x_cat = all_data[['dow', 'holiday_flag', 'is_precpt']].values.reshape(-1, 97, 3)

max_encoder_length = 90
holiday_label = 5

date_distance = 18 - x_cont[:, :max_encoder_length, 0][:, None, :] / 5
discount_distance = 1 - np.abs(x_cont[:, max_encoder_length:, 1][..., None] - x_cont[:, :max_encoder_length, 1][:, None, :])

decoder_dow = x_cat[:, max_encoder_length:, 0][..., None]
encoder_dow = x_cat[:, :max_encoder_length, 0][:, None, :]
dow_distance = decoder_dow - encoder_dow

decoder_precpt = x_cat[:, max_encoder_length:, 2][..., None]
encoder_precpt = x_cat[:, :max_encoder_length, 2][:, None, :]
precpt_distance = decoder_precpt - encoder_precpt

date_distance = np.where((decoder_dow == holiday_label).astype(int) - (encoder_dow == holiday_label).astype(int) == 0, date_distance - 3, date_distance)
date_distance = np.where((decoder_dow != holiday_label) & (encoder_dow == holiday_label), date_distance + 3, date_distance)
date_distance = np.where(dow_distance == 0, date_distance - 3, date_distance)
date_distance = np.where((precpt_distance == 0) & (decoder_precpt == 1), date_distance - 3, date_distance)
date_distance = np.where(precpt_distance != 0, date_distance + 3, date_distance)
date_distance = np.clip(date_distance, a_min=1, a_max=18)

del dow_distance, precpt_distance, decoder_precpt, encoder_precpt
weight = (18 - date_distance) * (discount_distance ** 2)
del date_distance, discount_distance
weight = np.exp(weight) / np.exp(weight).sum(axis=-1, keepdims=True)

# Tính psd từ sale_amount GỐC của tập train
mu = all_data[all_data['dt'] <= '2024-06-25'].groupby(['store_id', 'product_id'], as_index=False)['sale_amount'].mean()
mu.rename(columns={'sale_amount': 'psd'}, inplace=True)

pdf = all_data[all_data['dt'] >= '2024-06-26'].copy()
pdf = pdf.merge(mu, on=['store_id', 'product_id'], how='left')

target = x_cont[:, :max_encoder_length, 2:3]
index = encoder_dow.squeeze()[..., None] == np.arange(6)
mean_sale = np.nanmean(np.where(index, target, np.nan), axis=1) + 0.001
season_ratio = np.nanmedian(mean_sale[..., None] / mean_sale[:, None, ...], axis=0)
ratio = season_ratio[decoder_dow, encoder_dow]
pred = (weight * ratio) @ target
pdf['ssa_pred_sale_amount_pred'] = pred.reshape(-1)
del weight, ratio, x_cont, x_cat, all_data, pred

eligible_df = pdf.query('stock_hour6_22_cnt == 0')
eligible_rows = int(len(eligible_df))
low_sale_rows = int(len(eligible_df.query('psd < 1')))
high_sale_rows = int(len(eligible_df.query('psd >= 1')))
print(f'Kiểm tra mẫu hợp lệ: Overall = {eligible_rows:,} | Low Sale = {low_sale_rows:,} | High Sale = {high_sale_rows:,}')
assert eligible_rows == 206562, f'Số dòng hợp lệ ({eligible_rows}) khác 206562!'
assert low_sale_rows + high_sale_rows == eligible_rows

def evaluate_group(df_eval_pred, condition):
    wape_list, mae_list, wpe_list = [], [], []
    sub = df_eval_pred.query('stock_hour6_22_cnt == 0').query(condition)
    if sub['dt'].nunique() != 7:
        raise ValueError(f'{condition}: không đủ 7 ngày đánh giá.')
    for dt, subdf in sub.groupby('dt'):
        if subdf['sale_amount'].sum() <= 0 or not np.isfinite(subdf['ssa_pred_sale_amount_pred']).all():
            raise ValueError(f'{condition}, ngày {dt}: dự báo không hữu hạn hoặc tổng doanh số không dương.')
        abs_err = (subdf['sale_amount'] - subdf['ssa_pred_sale_amount_pred']).abs()
        wape_list.append(abs_err.sum() / subdf['sale_amount'].sum())
        mae_list.append(abs_err.mean())
        wpe_list.append((subdf['ssa_pred_sale_amount_pred'] - subdf['sale_amount']).sum() / subdf['sale_amount'].sum())
    return round(float(np.mean(wape_list)), 4), round(float(np.mean(wpe_list)), 4), round(float(np.mean(mae_list)), 4)

paper_ref_t11 = {
    'Overall':   {'cond': 'psd>=0', 'wape': 29.79, 'wpe': 1.39},
    'Low Sale':  {'cond': 'psd<1',  'wape': 37.87, 'wpe': 4.89},
    'High Sale': {'cond': 'psd>=1', 'wape': 23.98, 'wpe': -1.12},
}

log_lines = ['group	demand	wape	wpe	mae']
csv_rows = []

for group_name, info in paper_ref_t11.items():
    wape, wpe, mae = evaluate_group(pdf, info['cond'])
    log_lines.append(f"{info['cond']}	ssa_pred_sale_amount_pred	{wape}	{wpe}	{mae}")
    wape_pct = round(wape * 100, 2)
    wpe_pct = round(wpe * 100, 2)
    csv_rows.append({
        'config_id': 'SSA_TimesNet',
        'group': group_name,
        'wape_pct': wape_pct,
        'wpe_pct': wpe_pct,
        'mae': mae,
        'paper_wape_pct': info['wape'],
        'paper_wpe_pct': info['wpe'],
        'delta_wape_pp': round(wape_pct - info['wape'], 2),
        'delta_wpe_pp': round(wpe_pct - info['wpe'], 2),
        'status': 'completed',
    })

Kiểm tra mẫu hợp lệ: Overall = 206,562 | Low Sale = 153,208 | High Sale = 53,354


## 3. Lưu kết quả bàn giao và bảng đối chiếu song song với Task 8 (`SSA_RawSales`)

Lưu `ssa_timesnet_metrics.csv`, `ssa_raw_vs_timesnet_comparison.csv`, log tỷ lệ gốc `full/ssa_timesnet.log` và siêu dữ liệu môi trường `run_metadata.json`.

In [4]:
if len(csv_rows) != 3 or not np.isfinite(
        [[row[k] for k in ('wape_pct', 'wpe_pct', 'mae')] for row in csv_rows]).all():
    raise ValueError('Kết quả bàn giao phải có đúng ba dòng với chỉ số hữu hạn.')

(output_dir / 'full').mkdir(parents=True, exist_ok=True)
log_content = '\n'.join(log_lines) + '\n'
(output_dir / 'full/ssa_timesnet.log').write_text(log_content, encoding='utf-8')

df_t11 = pd.DataFrame(csv_rows)
df_t11.to_csv(output_dir / 'ssa_timesnet_metrics.csv', index=False, encoding='utf-8', lineterminator='\n')

if task08_csv is not None and task08_csv.is_file():
    df_t8 = pd.read_csv(task08_csv)
else:
    df_t8 = pd.DataFrame([
        {'config_id': 'SSA_RawSales', 'group': 'Overall',   'wape_pct': 31.97, 'wpe_pct': -10.50, 'mae': 0.3982, 'paper_wape_pct': 31.97, 'paper_wpe_pct': -10.50, 'delta_wape_pp': 0.0, 'delta_wpe_pp': 0.0, 'status': 'completed'},
        {'config_id': 'SSA_RawSales', 'group': 'Low Sale',  'wape_pct': 38.33, 'wpe_pct': -8.03,  'mae': 0.2691, 'paper_wape_pct': 38.33, 'paper_wpe_pct': -8.03,  'delta_wape_pp': 0.0, 'delta_wpe_pp': 0.0, 'status': 'completed'},
        {'config_id': 'SSA_RawSales', 'group': 'High Sale', 'wape_pct': 27.40, 'wpe_pct': -12.26, 'mae': 0.7761, 'paper_wape_pct': 27.40, 'paper_wpe_pct': -12.26, 'delta_wape_pp': 0.0, 'delta_wpe_pp': 0.0, 'status': 'completed'},
    ])

group_order = {'Overall': 0, 'Low Sale': 1, 'High Sale': 2}
df_compare = pd.concat([df_t8, df_t11], ignore_index=True)
df_compare['group_rank'] = df_compare['group'].map(group_order)
df_compare = df_compare.sort_values(by=['group_rank', 'config_id']).drop(columns=['group_rank']).reset_index(drop=True)
df_compare.to_csv(output_dir / 'ssa_raw_vs_timesnet_comparison.csv', index=False, encoding='utf-8', lineterminator='\n')

metadata = {
    'task_id': 'Task11',
    'config_id': 'SSA_TimesNet',
    'model': 'SSA',
    'imputation_source': 'TimesNet (Task 10)',
    'run_utc': datetime.now(timezone.utc).isoformat(),
    'runtime': 'Colab CPU' if drive is not None else f'{platform.system()} CPU',
    'python': platform.python_version(),
    'numpy': np.__version__,
    'pandas': pd.__version__,
    'pyarrow': pa.__version__,
    'repo_base_commit': 'bfff86f',
    'ssa_reference_commit': '8af694a09a5a427d1d50648d348c022565a781f2',
    'inputs': inputs,
    'target_column_used': 'sale_amount_pred',
    'psd_source_column': 'sale_amount (raw train)',
    'eval_ground_truth_column': 'sale_amount (raw eval)',
    'eligible_eval_rows': eligible_rows,
    'low_sale_eval_rows': low_sale_rows,
    'high_sale_eval_rows': high_sale_rows,
    'metrics_rows': len(df_t11),
    'metric_units': {'wape_pct': 'percent', 'wpe_pct': 'percent', 'mae': 'normalized sales'},
}
(output_dir / 'run_metadata.json').write_text(json.dumps(metadata, ensure_ascii=False, indent=2) + '\n', encoding='utf-8')

print('=== NỘI DUNG FILE LOG (ssa_timesnet.log) ===')
print(log_content, end='')
print('\n=== BẢNG ĐỐI CHIẾU SONG SONG 6 DÒNG (SSA_RawSales vs SSA_TimesNet) ===')
print(df_compare.to_string(index=False))

=== NỘI DUNG FILE LOG (ssa_timesnet.log) ===
group	demand	wape	wpe	mae
psd>=0	ssa_pred_sale_amount_pred	0.2979	0.014	0.3709
psd<1	ssa_pred_sale_amount_pred	0.3787	0.0496	0.2656
psd>=1	ssa_pred_sale_amount_pred	0.2398	-0.0115	0.679

=== BẢNG ĐỐI CHIẾU SONG SONG 6 DÒNG (SSA_RawSales vs SSA_TimesNet) ===
   config_id     group  wape_pct  wpe_pct    mae  paper_wape_pct  paper_wpe_pct  delta_wape_pp  delta_wpe_pp    status
SSA_RawSales   Overall     31.97   -10.50 0.3982           31.97         -10.50            0.0          0.00 completed
SSA_TimesNet   Overall     29.79     1.40 0.3709           29.79           1.39            0.0          0.01 completed
SSA_RawSales  Low Sale     38.33    -8.03 0.2691           38.33          -8.03            0.0          0.00 completed
SSA_TimesNet  Low Sale     37.87     4.96 0.2656           37.87           4.89            0.0          0.07 completed
SSA_RawSales High Sale     27.40   -12.26 0.7761           27.40         -12.26            0.0        